# FinSight RAG Evaluation Notebook

This notebook evaluates the RAG pipeline using RAGAS metrics.

In [ ]:
import requests
import json
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

## Configuration

In [ ]:
API_BASE = "http://localhost:8000/api/v1"

# Test questions for evaluation
test_questions = [
    "What was the total revenue for FY2023?",
    "What are the main risk factors mentioned?",
    "Who is the current CEO?",
    "What is the debt to equity ratio?",
]

## Run Queries and Collect Results

In [ ]:
results = []

for question in test_questions:
    resp = requests.post(f"{API_BASE}/query", json={"question": question})
    if resp.status_code == 200:
        data = resp.json()
        results.append({
            "question": question,
            "answer": data["answer"],
            "faithfulness": data["faithfulness_score"],
            "relevancy": data["answer_relevancy_score"],
            "fraud_score": data["fraud_risk_score"],
            "fraud_flags": data["fraud_flags"],
            "latency_ms": data["latency_ms"],
        })

df = pd.DataFrame(results)
df

## Summary Statistics

In [ ]:
print("=== RAG Evaluation Summary ===")
print(f"Average Faithfulness: {df['faithfulness'].mean():.3f}")
print(f"Average Relevancy: {df['relevancy'].mean():.3f}")
print(f"Average Latency: {df['latency_ms'].mean():.0f}ms")
print(f"Queries with Fraud Flags: {df[df['fraud_flags'].apply(len) > 0].shape[0]}")

## Visualize Metrics

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df["faithfulness"].plot(kind="bar", ax=axes[0], title="Faithfulness")
df["relevancy"].plot(kind="bar", ax=axes[1], title="Answer Relevancy")
df["latency_ms"].plot(kind="bar", ax=axes[2], title="Latency (ms)")

plt.tight_layout()
plt.show()